Library

In [103]:
import os
import pandas as pd
import pypdf
from pypdf import PdfReader
import pdfplumber
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string


# Load Text Data

In [104]:
from pathlib import Path

try:
    from pypdf import PdfReader
except Exception:
    PdfReader = None

personal_path = r"C:\Users\nhipk\OneDrive\0. Uni2024\Sem4\ML for Economists\FinalProject" #Edit your personal path here
folder_path = os.path.join(personal_path, "TermpaperFilesILIAS", "C_BundesbankMonthlyReports")
pdf_files = list(sorted(Path(folder_path).glob("*.pdf")))


In [105]:
def load_pdfs_from_folder(folder_path):
    """Read all PDF files from a specified folder, skip the first 4 and last 4 pages, and extract text."""
    data = []

    # Check if the target folder exists
    if not os.path.exists(folder_path):
        print(f"Error: The directory '{folder_path}' does not exist.")
        return None

    # Iterate through all items within the folder
    for filename in os.listdir(folder_path):
        # Process only files with a .pdf extension (case-insensitive)
        if filename.lower().endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            print(f"Processing: {filename}...")

            try:
                # Initialize the PDF reader
                reader = PdfReader(file_path)
                total_pages = len(reader.pages)
                
                # Safe check: Only process if the file has more than 8 pages
                if total_pages > 8:
                    full_text = ""
                    
                    # Slice the pages to skip the first 4 and last 4 pages
                    target_pages = reader.pages[4:-4]
                    
                    # Loop through the sliced pages to extract text content
                    for page in target_pages:
                        text = page.extract_text()
                        if text:
                            full_text += text + "\n"

                    # Append the filename and extracted text metadata to the list
                    data.append({"file_name": filename, "text": full_text.strip()})
                else:
                    print(f"⚠️ Skipped '{filename}': File only has {total_pages} pages (too short to drop 4 start/end pages).")

            except Exception as e:
                print(f"Error reading file {filename}: {e}")

    # Convert the extracted data into a pandas DataFrame for seamless NLP/Sentiment Analysis
    df = pd.DataFrame(data)
    return df

# Execute the extraction process
df_sentiment = load_pdfs_from_folder(folder_path)

# Display a preview of the resulting DataFrame
if df_sentiment is not None and not df_sentiment.empty:
    print(f"\nSuccessfully loaded {len(df_sentiment)} files!")
    display(df_sentiment)
else:
    print("\nNo PDF files found or the folder is empty.")

Processing: 2009-12-monatsbericht-data.pdf...
Processing: 2010-12-monatsbericht-data.pdf...
Processing: 2011-12-monatsbericht-data.pdf...
Processing: 2012-12-monatsbericht-data.pdf...
Processing: 2013-12-monatsbericht-data.pdf...
Processing: 2014-12-monatsbericht-data.pdf...
Processing: 2015-12-monatsbericht-data.pdf...
Processing: 2016-12-monatsbericht-data.pdf...
Processing: 2017-12-monatsbericht-data.pdf...
Processing: 2018-12-monatsbericht-data.pdf...
Processing: 2019-12-monatsbericht-data.pdf...
Processing: 2020-12-monatsbericht-data.pdf...
Processing: 2021-12-monatsbericht-data.pdf...
Processing: 2022-12-monatsbericht-data.pdf...
Processing: 2023-12-monatsbericht-data.pdf...

Successfully loaded 15 files!


,file_name,text
0,2009-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...
1,2010-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...
2,2011-12-monatsbericht-data.pdf,Commentaries Economic conditions\nUnderlying t...
3,2012-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
4,2013-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
5,2014-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
6,2015-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
7,2016-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
8,2017-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
9,2018-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...


# Process Data

In [106]:
# Add date column as key
df_sentiment["report_date"] = (
    df_sentiment["file_name"].str[:7]
    )
df_sentiment["word_count"] = df_sentiment["text"].str.split().str.len()

display(df_sentiment)


,file_name,text,report_date,word_count
0,2009-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...,2009-12,100371
1,2010-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...,2010-12,102536
2,2011-12-monatsbericht-data.pdf,Commentaries Economic conditions\nUnderlying t...,2011-12,113864
3,2012-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2012-12,119657
4,2013-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2013-12,113182
5,2014-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2014-12,105413
6,2015-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2015-12,112959
7,2016-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2016-12,120586
8,2017-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2017-12,127829
9,2018-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2018-12,116731


## Text Cleaning

In [107]:
# Clean white spaces, convert to lowercase and remove punctuation
punctuation_pattern = f"[{''.join([f'\\{c}' for c in string.punctuation])}]"
df_sentiment['cleaned_text'] = (
    df_sentiment['text']
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
    .str.replace(punctuation_pattern, '', regex=True)
)

# Drop file_name and full text columns to keep the DataFrame clean for further analysis
df_sentiment = df_sentiment.drop(columns=["file_name", "text"])

## Tokenize Text

In [108]:
# Tokenize cleaned text
df_sentiment['tokens'] = df_sentiment['cleaned_text'].apply(word_tokenize)

# Token counts
df_sentiment["token_count"] = df_sentiment["tokens"].apply(len)
display(df_sentiment)

,report_date,word_count,cleaned_text,tokens,token_count
0,2009-12,100371,deutsche bundesbank monthly report december 20...,"[deutsche, bundesbank, monthly, report, decemb...",95111
1,2010-12,102536,deutsche bundesbank monthly report december 20...,"[deutsche, bundesbank, monthly, report, decemb...",97156
2,2011-12,113864,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",108312
3,2012-12,119657,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",114115
4,2013-12,113182,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",108032
5,2014-12,105413,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",99657
6,2015-12,112959,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",106915
7,2016-12,120586,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",114137
8,2017-12,127829,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",121182
9,2018-12,116731,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",110304


## Remove Stop Words and Custom Words

In [109]:
# Stop words and Custom Words
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

# Load standard English stop words
stop_words = set(stopwords.words("english"))
print(stop_words)


{'needn', "i'm", 'my', 'no', 'of', "she'd", 'same', 'these', 'very', 'y', 'herself', 'ourselves', "you've", 'hasn', 'but', "it's", 'now', "mustn't", 're', 'he', "we'll", "we'd", 'ours', 'why', 'such', "we've", 'or', "it'll", 'mightn', 'in', 'be', 'we', "he's", 'she', 'were', 'there', 'can', "haven't", "he'll", 'from', 'over', 'yourself', 'because', 'don', 'aren', 'ma', 'an', 'did', "weren't", "hadn't", "shouldn't", 'yourselves', 'by', 'shan', 'if', 'are', 'your', 'you', 'this', "you'll", 've', 'against', 'ain', "she'll", 'theirs', 'being', "it'd", "we're", 'to', 'after', 'itself', "i'll", 'that', 'had', 'where', 'weren', 'on', 'whom', 'shouldn', 'between', 'down', 'didn', 'doing', "isn't", 'most', 'his', 'off', 't', 'own', 'haven', 'with', 'above', 'hers', 'd', 'into', "they'll", 'wouldn', "don't", 'been', 'about', 'for', 'is', 'him', 'o', 'am', "mightn't", "you'd", 'them', 'its', 'during', 'again', "shan't", 'a', 'will', "didn't", 'how', 'her', 'then', 'should', 'until', 'through', 'a

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [110]:
# A list of custom words to be removed
custom_stop_words = {
    "report", "monthly", "bundesbank", "deutsche", "german", "germany", "country", "countries"
    "central", "bank", "european", "euro", "zone", "europe", 
    "ecb", "monetary", "policy","december","commentaries", "monatsbericht", "bericht", "bundesbankbericht", "bundesbankmonatsbericht", "bundesbankmonatsberichte", "bundesbankmonatsberichten", "bundesbankmonatsberichts", "bundesbankmonatsberichten", "bundesbankmonatsberichts", "bundesbankmonatsberichten", "bundesbankmonatsberichts", "bundesbankmonatsberichten", "bundesbankmonatsberichts", "bundesbankmonatsberichten",
    "data", "page", "figure", "chart", "appendix", "source", "statist"
}
all_stop_words = custom_stop_words.union(stop_words)

# Remove stop words
df_sentiment['tokens'] = df_sentiment['tokens'].apply(lambda x: [word for word in x if word not in all_stop_words])

# Remove short tokens, tokens with numbers, and tokens with underscores or special characters
def apply_regex_vocabulary_filter(token_list):
    """
    Filter a list of tokens using strict regex and string rules:
    - Removes tokens shorter than 3 characters (e.g., 'aa', 'ab').
    - Removes tokens containing any numbers/digits (e.g., 'a1', 'a15').
    - Removes tokens containing underscores or special metadata symbols (e.g., '_page_').
    """
    if not isinstance(token_list, list):
        return []
        
    cleaned_tokens = []
    for token in token_list:
        token = str(token).strip() # Ensure the token is a valid string and strip any spaces
        
        if len(token) < 3: # Check if token length is less than 3
            continue
            
        if any(char.isdigit() for char in token): # Check if token contains any numerical digits
            continue
            
        if not re.match(r'^[a-zA-Z]+$', token): # Check if token contains underscores or non-alphabetic characters

            continue
            
        # If the token passes all the criteria above, keep it
        cleaned_tokens.append(token)
        
    return cleaned_tokens

# Filter the original tokens column using the pattern-based rules
df_sentiment['tokens'] = df_sentiment['tokens'].apply(apply_regex_vocabulary_filter)

df_sentiment["token_count"] = df_sentiment["tokens"].str.len()

display(df_sentiment)

,report_date,word_count,cleaned_text,tokens,token_count
0,2009-12,100371,deutsche bundesbank monthly report december 20...,"[economic, conditions, underlying, trends, rec...",26932
1,2010-12,102536,deutsche bundesbank monthly report december 20...,"[economic, conditions, underlying, trends, upt...",27626
2,2011-12,113864,commentaries economic conditions underlying tr...,"[economic, conditions, underlying, trends, acc...",29894
3,2012-12,119657,commentaries economic conditions underlying tr...,"[economic, conditions, underlying, trends, eco...",31538
4,2013-12,113182,commentaries economic conditions underlying tr...,"[economic, conditions, underlying, trends, des...",33215
5,2014-12,105413,commentaries economic conditions underlying tr...,"[economic, conditions, underlying, trends, eco...",27218
6,2015-12,112959,commentaries economic conditions underlying tr...,"[economic, conditions, underlying, trends, eco...",31642
7,2016-12,120586,commentaries economic conditions underlying tr...,"[economic, conditions, underlying, trends, eco...",35058
8,2017-12,127829,commentaries economic conditions underlying tr...,"[economic, conditions, underlying, trends, str...",38709
9,2018-12,116731,commentaries economic conditions underlying tr...,"[economic, conditions, underlying, trends, exp...",33120


## Part-of-Speech (POS) Tagging and Filtering

In [111]:
nltk.download('averaged_perceptron_tagger_eng')

def pos_tag_and_filter(token_list):
    """
    Perform POS Tagging on a list of tokens and filter to keep only 
    specific parts of speech (Nouns, Verbs, Adjectives, Adverbs).
    """
    if not isinstance(token_list, list) or len(token_list) == 0:
        return []
    
    # 1. Apply POS Tagging to the list of tokens
    # Output is a list of tuples: [('token', 'TAG'), ...]
    tagged_tokens = nltk.pos_tag(token_list)
    
    # 2. Define the target POS tags we want to keep
    # N: Nouns (NN, NNS, NNP, NNPS)
    # V: Verbs (VB, VBD, VBG, VBN, VBP, VBZ)
    # J: Adjectives (JJ, JJR, JJS)
    # R: Adverbs (RB, RBR, RBS)
    target_prefixes = ('N', 'V', 'J', 'R')
    
    # 3. Filter tokens based on their POS tags
    filtered_tokens = [
        token for token, tag in tagged_tokens
        if tag.startswith(target_prefixes)
    ]
    
    return filtered_tokens

# Compute POS-tagged and filtered tokens, storing
df_sentiment['tokens'] = df_sentiment['tokens'].apply(pos_tag_and_filter)
df_sentiment["token_count"] = df_sentiment["tokens"].apply(len)

# Reconstruct the cleaned text from the filtered tokens
df_sentiment['cleaned_text'] = df_sentiment['tokens'].apply(lambda x: " ".join(x))


df_sentiment["word_count"] = df_sentiment["cleaned_text"].str.split().str.len()

display(df_sentiment)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


,report_date,word_count,cleaned_text,tokens,token_count
0,2009-12,25915,economic conditions underlying trends recovery...,"[economic, conditions, underlying, trends, rec...",25915
1,2010-12,26634,economic conditions underlying trends upturn e...,"[economic, conditions, underlying, trends, upt...",26634
2,2011-12,28854,economic conditions underlying trends accordin...,"[economic, conditions, underlying, trends, acc...",28854
3,2012-12,30439,economic conditions underlying trends economic...,"[economic, conditions, underlying, trends, eco...",30439
4,2013-12,32037,economic conditions underlying trends modest s...,"[economic, conditions, underlying, trends, mod...",32037
5,2014-12,26298,economic conditions underlying trends economy ...,"[economic, conditions, underlying, trends, eco...",26298
6,2015-12,30487,economic conditions underlying trends economy ...,"[economic, conditions, underlying, trends, eco...",30487
7,2016-12,33857,economic conditions underlying trends economic...,"[economic, conditions, underlying, trends, eco...",33857
8,2017-12,37411,economic conditions underlying trends strong u...,"[economic, conditions, underlying, trends, str...",37411
9,2018-12,31964,economic conditions underlying trends experien...,"[economic, conditions, underlying, trends, exp...",31964


## N-grams Processing

In [112]:
from nltk.util import ngrams

def generate_ngrams(token_list, n=2):
    """
    Generate N-grams from a list of tokens.
    Returns a list of space-separated N-gram strings.
    """
    if not isinstance(token_list, list) or len(token_list) < n:
        return []
    
    # 1. Use NLTK's ngrams utility to create tuples of size n
    # Output: [('long', 'term'), ('term', 'debt')]
    ngram_tuples = list(ngrams(token_list, n))
    
    # 2. Join the tuples with underscores or spaces to represent single features
    # Standard NLP convention often uses underscore (e.g., 'long_term') 
    # to keep them as single tokens in downstream TF-IDF
    ngram_strings = ["_".join(words) for words in ngram_tuples]
    
    return ngram_strings


df_sentiment['bigrams'] = df_sentiment['tokens'].apply(lambda x: generate_ngrams(x, n=2)) # Generate Bigrams (2-grams)


df_sentiment['trigrams'] = df_sentiment['tokens'].apply(lambda x: generate_ngrams(x, n=3)) # Generate Trigrams (3-grams)

df_sentiment['all_ngram_features'] = df_sentiment.apply(
    lambda row: row['tokens'] + row['bigrams'] + row['trigrams'], 
    axis=1
) # Combine Unigrams

df_sentiment['cleaned_text'] = df_sentiment['all_ngram_features'].apply(lambda tokens: ' '.join(tokens))

display(df_sentiment)

,report_date,word_count,cleaned_text,tokens,token_count,bigrams,trigrams,all_ngram_features
0,2009-12,25915,economic conditions underlying trends recovery...,"[economic, conditions, underlying, trends, rec...",25915,"[economic_conditions, conditions_underlying, u...","[economic_conditions_underlying, conditions_un...","[economic, conditions, underlying, trends, rec..."
1,2010-12,26634,economic conditions underlying trends upturn e...,"[economic, conditions, underlying, trends, upt...",26634,"[economic_conditions, conditions_underlying, u...","[economic_conditions_underlying, conditions_un...","[economic, conditions, underlying, trends, upt..."
2,2011-12,28854,economic conditions underlying trends accordin...,"[economic, conditions, underlying, trends, acc...",28854,"[economic_conditions, conditions_underlying, u...","[economic_conditions_underlying, conditions_un...","[economic, conditions, underlying, trends, acc..."
3,2012-12,30439,economic conditions underlying trends economic...,"[economic, conditions, underlying, trends, eco...",30439,"[economic_conditions, conditions_underlying, u...","[economic_conditions_underlying, conditions_un...","[economic, conditions, underlying, trends, eco..."
4,2013-12,32037,economic conditions underlying trends modest s...,"[economic, conditions, underlying, trends, mod...",32037,"[economic_conditions, conditions_underlying, u...","[economic_conditions_underlying, conditions_un...","[economic, conditions, underlying, trends, mod..."
5,2014-12,26298,economic conditions underlying trends economy ...,"[economic, conditions, underlying, trends, eco...",26298,"[economic_conditions, conditions_underlying, u...","[economic_conditions_underlying, conditions_un...","[economic, conditions, underlying, trends, eco..."
6,2015-12,30487,economic conditions underlying trends economy ...,"[economic, conditions, underlying, trends, eco...",30487,"[economic_conditions, conditions_underlying, u...","[economic_conditions_underlying, conditions_un...","[economic, conditions, underlying, trends, eco..."
7,2016-12,33857,economic conditions underlying trends economic...,"[economic, conditions, underlying, trends, eco...",33857,"[economic_conditions, conditions_underlying, u...","[economic_conditions_underlying, conditions_un...","[economic, conditions, underlying, trends, eco..."
8,2017-12,37411,economic conditions underlying trends strong u...,"[economic, conditions, underlying, trends, str...",37411,"[economic_conditions, conditions_underlying, u...","[economic_conditions_underlying, conditions_un...","[economic, conditions, underlying, trends, str..."
9,2018-12,31964,economic conditions underlying trends experien...,"[economic, conditions, underlying, trends, exp...",31964,"[economic_conditions, conditions_underlying, u...","[economic_conditions_underlying, conditions_un...","[economic, conditions, underlying, trends, exp..."


## Document-term matrix (DTM)

$$\text{TF}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$$

$$\text{IDF}(t) = \ln \left( \frac{1 + N}{1 + \text{DF}(t)} \right) + 1$$

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

In [113]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(min_df=1, use_idf=True, smooth_idf=True)

# Create DTM
dtm_sparse = tfidf_vectorizer.fit_transform(df_sentiment['cleaned_text'])

# Retrieve the feature names (vocabulary) corresponding to the columns of the matrix
feature_names = tfidf_vectorizer.get_feature_names_out()

df_dtm = pd.DataFrame(
    dtm_sparse.toarray(), 
    columns=feature_names, 
    index=df_sentiment.index
)

# Combine DTM with original metadata for analysis
df_dtm = pd.concat([df_sentiment[['report_date']], df_dtm], axis=1)

print("Document-Term Matrix (DTM) successfully generated!")
display(df_dtm)

Document-Term Matrix (DTM) successfully generated!


,report_date,aaarated,aaarated_countries,aaarated_countries_united,aapor,aapor_defi,aapor_defi_wave,aapor_response,aapor_response_rate,abandoned,...,zvs_working_group,zweites,zweites_including,zweites_including_fundamental,zwischen,zwischen_der,zwischen_der_sozialdemokratischen,zzle,zzle_form,zzle_form_additional
0,2009-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002043,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2010-12,0.001857,0.001857,0.001857,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2011-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2012-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2013-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,2014-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,2015-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.001774,0.001774,0.001774,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,2016-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,2017-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00325,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,2018-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [ ]:
# Compress DTM using TruncatedSVD (LSA) to reduce dimensionality and extract semantic topics
from sklearn.decomposition import TruncatedSVD

# Set report_date as the index for the sentiment results
df_dtm = df_dtm.set_index('report_date')

# Dimensionality Reduction of the massive DTM using TruncatedSVD (LSA)
# Compressing sparse word columns into 5 dense semantic topics
n_topics = 5
svd = TruncatedSVD(n_components=n_topics, random_state=42)
dtm_reduced = svd.fit_transform(df_dtm)

# Convert the reduced matrix back into a structured DataFrame
component_names = [f"text_concept_{i}" for i in range(n_topics)]
df_text_concepts = pd.DataFrame(
    dtm_reduced, 
    columns=component_names, 
    index=df_dtm.index
)
display(df_text_concepts)


,text_concept_0,text_concept_1,text_concept_2,text_concept_3,text_concept_4
report_date,,,,,
2009-12,0.879991,-0.093056,-0.146153,0.163875,-0.039970
2010-12,0.835960,-0.189447,-0.203610,0.379189,-0.067796
2011-12,0.879532,-0.298968,0.197458,-0.000322,-0.016279
2012-12,0.828105,-0.367395,0.346822,-0.085331,-0.011707
2013-12,0.884474,-0.088450,-0.100576,0.046451,0.096496
2014-12,0.935348,-0.037014,-0.069417,-0.048745,0.029641
2015-12,0.871245,-0.011428,-0.144722,-0.121441,0.115964
2016-12,0.898066,0.023083,-0.097797,-0.123634,0.100084
2017-12,0.872840,0.040744,-0.107276,-0.131699,0.157272


In [ ]:
# [TODO] Extract key words for each topic from the SVD components

## Positive/Negative Polarity

In [115]:
import pysentiment2 as ps

# Initialize the Loughran-McDonald financial dictionary
lm = ps.LM()

positive_words = list(lm._posset)
negative_words = list(lm._negset)

# Print the total counts and a small sample
print(f"Total Positive Words (Stemmed): {len(positive_words)}")
print(f"Sample Positive Words: {sorted(positive_words)[:15]}\n")

print(f"Total Negative Words (Stemmed): {len(negative_words)}")
print(f"Sample Negative Words: {sorted(negative_words)[:15]}")

Total Positive Words (Stemmed): 140
Sample Positive Words: ['abund', 'acclaim', 'accomplish', 'achiev', 'adequ', 'advanc', 'advantag', 'allianc', 'assur', 'attain', 'attract', 'beauti', 'benefici', 'benefit', 'better']

Total Negative Words (Stemmed): 893
Sample Negative Words: ['abandon', 'abdic', 'aberr', 'abet', 'abnorm', 'abolish', 'abrog', 'abrupt', 'abruptli', 'absenc', 'absente', 'abus', 'accid', 'accident', 'accus']


In [116]:
# To check if a word is in the positive or negative list, you can use the following function:
def check_word_sentiment(word):
    """Check if a word is in the positive or negative sentiment lists."""
    if word in positive_words:
        return "Positive"
    elif word in negative_words:
        return "Negative"
    else:
        return "Neutral"

check_word_sentiment("down")  

'Neutral'

$$\text{Polarity} = \frac{\text{Pos} - \text{Neg}}{\text{Pos} + \text{Neg}}$$

In [117]:
# Extract Positive and Negative counts
def get_financial_sentiment(text):
    tokens = lm.tokenize(text)
    score = lm.get_score(tokens)
    
    return score['Positive'], score['Negative'], score['Polarity']


# We zip the results into three new columns
df_sentiment['lm_positive'], df_sentiment['lm_negative'], df_sentiment['lm_polarity'] = zip(
    *df_sentiment['cleaned_text'].apply(get_financial_sentiment)
)


display(df_sentiment[['report_date', 'word_count', 'lm_positive', 'lm_negative', 'lm_polarity']])

,report_date,word_count,lm_positive,lm_negative,lm_polarity
0,2009-12,25915,2790,3894,-0.165171
1,2010-12,26634,2712,4818,-0.279681
2,2011-12,28854,2592,4374,-0.255814
3,2012-12,30439,3348,4260,-0.119874
4,2013-12,32037,3354,5196,-0.215439
5,2014-12,26298,2436,3828,-0.222222
6,2015-12,30487,2838,3930,-0.161348
7,2016-12,33857,3396,5808,-0.262060
8,2017-12,37411,3534,4980,-0.169838
9,2018-12,31964,3282,4434,-0.149300


# Combine and extract input data for modelling

In [ ]:
# Input code here